In [ ]:
# --- Deep learning (may be needed downstream) ---
from torch.utils.data import DataLoader
import torch
from torchvision import transforms

# --- Data processing ---
import os
import numpy as np
import pandas as pd
from pathlib import Path

# --- Image processing ---
from PIL import ImageFile, Image
Image.MAX_IMAGE_PIXELS = None
from skimage import io, img_as_float32
import matplotlib.pyplot as plt

# --- Whole-slide image handling ---
import openslide
from openslide import OpenSlide

# --- Spatial transcriptomics ---
import scanpy as sc
from sklearn.neighbors import NearestNeighbors

# --- Pretrained models ---
from transformers import AutoImageProcessor, AutoModel

# --- Custom modules ---
from finetune.vision_transformer import vit_small

## SVS → JPEG Conversion

Converts TCGA Aperio SVS whole-slide images to JPEG thumbnails using OpenSlide.
Uses the second-lowest resolution level for a balance of file size and image quality.

In [ ]:
TCGA_img_dir = r'./实验/实验数据/TCGA-BRCA数据/Slide_Image/'

for name in os.listdir(TCGA_img_dir):
    sample_dir = os.path.join(TCGA_img_dir, name)
    if not os.path.isdir(sample_dir):
        continue  # Skip non-folders (e.g., .ipynb_checkpoints)

    svs_path = os.path.join(sample_dir, f'{name}.svs')
    if not os.path.exists(svs_path):
        print(f"SKIP {name}: SVS not found at {svs_path}")
        continue

    try:
        slide = OpenSlide(svs_path)
        level_count = slide.level_count

        # Use second-lowest resolution for the thumbnail
        thumbnail_level = max(0, level_count - 2)
        thumbnail = slide.read_region(
            (0, 0),
            thumbnail_level,
            slide.level_dimensions[thumbnail_level]
        )
        thumbnail_rgb = thumbnail.convert('RGB')

        output_path = os.path.join(sample_dir, f'{name}.jpg')
        thumbnail_rgb.save(output_path, 'JPEG', quality=95)
        print(f"OK: {output_path}")

        slide.close()

    except Exception as e:
        print(f"ERROR processing {name}: {e}")

### Optional: Visualize a single thumbnail

In [ ]:
# slide = OpenSlide(svs_path)
# level = slide.level_count - 2
# thumbnail = slide.read_region((0, 0), level, slide.level_dimensions[level])
# plt.figure(figsize=(12, 10))
# plt.imshow(thumbnail)
# plt.axis('off')
# slide.close()
# plt.show()